In [1]:
import pandas as pd
import numpy as np

In [2]:
batch_date = pd.Timestamp("2026-09-30")

customer_dim = pd.DataFrame({
    "customer_id": [
        "C101", "C101",
        "C102",
        "C103", "C103",
        "C104",
        "C105"
    ],
    "city": [
        "Mumbai", "Pune",
        "Delhi",
        "Bangalore", "Chennai",
        "Hyderabad",
        "Kolkata"
    ],
    "segment": [
        "Silver", "Gold",
        "Silver",
        "Bronze", "Silver",
        "Gold",
        "Bronze"
    ],
    "effective_start": pd.to_datetime([
        "2025-01-10", "2026-03-15",
        "2025-11-20",
        "2025-06-01", "2026-05-10",
        "2026-01-25",
        "2026-07-01"
    ]),
    "effective_end": pd.to_datetime([
        "2026-03-14", None,
        None,
        "2026-05-09", None,
        None,
        None
    ]),
    "is_current": [
        False, True,
        True,
        False, True,
        True,
        True
    ]
})

incoming_customers = pd.DataFrame({
    "customer_id": ["C101", "C102", "C103", "C104", "C106"],
    "city": [
        "Pune",
        "Mumbai",
        "Chennai",
        "Hyderabad",
        "Ahmedabad"
    ],
    "segment": [
        "Gold",
        "Silver",
        "Gold",
        "Gold",
        "Silver"
    ]
})

In [3]:
customer_dim

,customer_id,city,segment,effective_start,effective_end,is_current
0,C101,Mumbai,Silver,2025-01-10,2026-03-14,False
1,C101,Pune,Gold,2026-03-15,NaT,True
2,C102,Delhi,Silver,2025-11-20,NaT,True
3,C103,Bangalore,Bronze,2025-06-01,2026-05-09,False
4,C103,Chennai,Silver,2026-05-10,NaT,True
5,C104,Hyderabad,Gold,2026-01-25,NaT,True
6,C105,Kolkata,Bronze,2026-07-01,NaT,True


In [4]:
incoming_customers

,customer_id,city,segment
0,C101,Pune,Gold
1,C102,Mumbai,Silver
2,C103,Chennai,Gold
3,C104,Hyderabad,Gold
4,C106,Ahmedabad,Silver


Q1 - Current customers from customer dim

In [5]:
customer_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   customer_id      7 non-null      str           
 1   city             7 non-null      str           
 2   segment          7 non-null      str           
 3   effective_start  7 non-null      datetime64[us]
 4   effective_end    2 non-null      datetime64[us]
 5   is_current       7 non-null      bool          
dtypes: bool(1), datetime64[us](2), str(3)
memory usage: 419.0 bytes


In [6]:
current_customers = customer_dim[customer_dim['is_current'] == True]

In [7]:
current_customers

,customer_id,city,segment,effective_start,effective_end,is_current
1,C101,Pune,Gold,2026-03-15,NaT,True
2,C102,Delhi,Silver,2025-11-20,NaT,True
4,C103,Chennai,Silver,2026-05-10,NaT,True
5,C104,Hyderabad,Gold,2026-01-25,NaT,True
6,C105,Kolkata,Bronze,2026-07-01,NaT,True


Q2 — Reconciliation \
Reconcile incoming_customers against current_customers using customer_id.

In [8]:
joined = incoming_customers.merge(
    current_customers,
    on='customer_id',
    how='left',
    indicator=True,
    suffixes=('_incoming', '_current')
)

In [9]:
joined

,customer_id,city_incoming,segment_incoming,city_current,segment_current,effective_start,effective_end,is_current,_merge
0,C101,Pune,Gold,Pune,Gold,2026-03-15,NaT,True,both
1,C102,Mumbai,Silver,Delhi,Silver,2025-11-20,NaT,True,both
2,C103,Chennai,Gold,Chennai,Silver,2026-05-10,NaT,True,both
3,C104,Hyderabad,Gold,Hyderabad,Gold,2026-01-25,NaT,True,both
4,C106,Ahmedabad,Silver,NaN,NaN,NaT,NaT,NaN,left_only


Q3 — Find New Customers

In [10]:
new_customers = joined[joined['_merge'] == 'left_only']

In [11]:
existing_customers = joined[joined['_merge'] == 'both']

In [12]:
new_customers

,customer_id,city_incoming,segment_incoming,city_current,segment_current,effective_start,effective_end,is_current,_merge
4,C106,Ahmedabad,Silver,NaN,NaN,NaT,NaT,NaN,left_only


In [13]:
existing_customers

,customer_id,city_incoming,segment_incoming,city_current,segment_current,effective_start,effective_end,is_current,_merge
0,C101,Pune,Gold,Pune,Gold,2026-03-15,NaT,True,both
1,C102,Mumbai,Silver,Delhi,Silver,2025-11-20,NaT,True,both
2,C103,Chennai,Gold,Chennai,Silver,2026-05-10,NaT,True,both
3,C104,Hyderabad,Gold,Hyderabad,Gold,2026-01-25,NaT,True,both


Q4 — Find Changed Customers

In [14]:
changed_customers = existing_customers[(existing_customers['city_current'] != existing_customers['city_incoming']) | (existing_customers['segment_incoming'] != existing_customers['segment_current'])]

In [15]:
changed_customers

,customer_id,city_incoming,segment_incoming,city_current,segment_current,effective_start,effective_end,is_current,_merge
1,C102,Mumbai,Silver,Delhi,Silver,2025-11-20,NaT,True,both
2,C103,Chennai,Gold,Chennai,Silver,2026-05-10,NaT,True,both


Q5 — Find Unchanged Customers

In [16]:
unchanged_customers = existing_customers[~(existing_customers['customer_id'].isin(changed_customers['customer_id']))]

In [17]:
unchanged_customers

,customer_id,city_incoming,segment_incoming,city_current,segment_current,effective_start,effective_end,is_current,_merge
0,C101,Pune,Gold,Pune,Gold,2026-03-15,NaT,True,both
3,C104,Hyderabad,Gold,Hyderabad,Gold,2026-01-25,NaT,True,both


### SCD Project

In [18]:
pd.Timestamp.now().normalize()

Timestamp('2026-09-30 00:00:00')

In [19]:
current_date = pd.Timestamp.now().normalize()

In [20]:
current_date

Timestamp('2026-09-30 00:00:00')

In [21]:
changed_customers = changed_customers.assign(
    is_current=False,
    effective_end=current_date - pd.Timedelta(days=1)
)

In [22]:
new_versions = changed_customers.copy()

In [23]:
new_versions

,customer_id,city_incoming,segment_incoming,city_current,segment_current,effective_start,effective_end,is_current,_merge
1,C102,Mumbai,Silver,Delhi,Silver,2025-11-20,2026-09-29,False,both
2,C103,Chennai,Gold,Chennai,Silver,2026-05-10,2026-09-29,False,both


In [24]:
new_versions['effective_start'] = current_date

In [25]:
new_versions['effective_end'] = pd.NaT

In [26]:
new_versions['is_current'] = True

In [27]:
new_versions = new_versions.loc[
    :,
    [
        'customer_id',
        'city_incoming',
        'segment_incoming',
        'effective_start',
        'effective_end',
        'is_current'
    ]
].rename(
    columns={
        'city_incoming': 'city',
        'segment_incoming': 'segment'
    }
)

In [28]:
new_versions

,customer_id,city,segment,effective_start,effective_end,is_current
1,C102,Mumbai,Silver,2026-09-30,NaT,True
2,C103,Chennai,Gold,2026-09-30,NaT,True


In [29]:
new_customers = new_customers.loc[
    :,
    [
        'customer_id',
        'city_incoming',
        'segment_incoming',
        'effective_start',
        'effective_end',
        'is_current'
    ]
].rename(
    columns={
        'city_incoming': 'city',
        'segment_incoming': 'segment'
    }
)

In [30]:
new_customers['effective_start'] = current_date
new_customers['effective_end'] = pd.NaT
new_customers['is_current'] = True

In [31]:
new_customers

,customer_id,city,segment,effective_start,effective_end,is_current
4,C106,Ahmedabad,Silver,2026-09-30,NaT,True


In [32]:
pd.concat([customer_dim, new_versions, new_customers], ignore_index=True)

,customer_id,city,segment,effective_start,effective_end,is_current
0,C101,Mumbai,Silver,2025-01-10,2026-03-14,False
1,C101,Pune,Gold,2026-03-15,NaT,True
2,C102,Delhi,Silver,2025-11-20,NaT,True
3,C103,Bangalore,Bronze,2025-06-01,2026-05-09,False
4,C103,Chennai,Silver,2026-05-10,NaT,True
5,C104,Hyderabad,Gold,2026-01-25,NaT,True
6,C105,Kolkata,Bronze,2026-07-01,NaT,True
7,C102,Mumbai,Silver,2026-09-30,NaT,True
8,C103,Chennai,Gold,2026-09-30,NaT,True
9,C106,Ahmedabad,Silver,2026-09-30,NaT,True
